# 02 - Data Cleaning: El Clásico Dataset

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# Load the raw data
df = pd.read_csv('../data/raw/wikipedia_el_clasico_bs4.csv')
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

In [ ]:
# Initial data inspection
print("Data Types:")
print(df.dtypes)
print("\n" + "="*50)
print("\nMissing Values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("\nBasic Statistics:")
df.describe()

---

## 1. Deduplication

Remove duplicate records.

In [ ]:
# Check for duplicates
print(f"Total rows: {len(df)}")
print(f"Duplicate rows: {df.duplicated().sum()}")

# Remove duplicates
df_clean = df.drop_duplicates()
print(f"Rows after deduplication: {len(df_clean)}")

---

## 2. Handling Missing Data

### 2.1 Simple Imputation

In [ ]:
# Identify missing values
missing_summary = df_clean.isnull().sum()
missing_pct = (df_clean.isnull().sum() / len(df_clean)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_summary,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

print("Missing Value Summary:")
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# Simple Imputation for numeric columns
# Using mean, median, or most_frequent

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

# Mean imputation
mean_imputer = SimpleImputer(strategy='mean')

# Median imputation (better for skewed data)
median_imputer = SimpleImputer(strategy='median')

# Mode imputation for categorical
mode_imputer = SimpleImputer(strategy='most_frequent')

In [ ]:
# Example: Apply simple imputation
# df_clean[numeric_cols] = mean_imputer.fit_transform(df_clean[numeric_cols])

### 2.2 KNN Imputation

In [ ]:
# KNN Imputation - uses similar records to fill missing values
knn_imputer = KNNImputer(n_neighbors=5)

# Example usage:
# df_knn = df_clean.copy()
# df_knn[numeric_cols] = knn_imputer.fit_transform(df_knn[numeric_cols])

print("KNN Imputer configured with k=5 neighbors")

### 2.3 Iterative Imputation - MICE

In [ ]:
# MICE - Multiple Imputation by Chained Equations
mice_imputer = IterativeImputer(max_iter=10, random_state=42)

# Example usage:
# df_mice = df_clean.copy()
# df_mice[numeric_cols] = mice_imputer.fit_transform(df_mice[numeric_cols])

print("MICE Imputer configured with max_iter=10")

---

## 3. Data Transformation

### 3.1 Continuous to Categorical Conversion

In [ ]:
# Example: Convert total goals to categories
# Assuming we have a 'total_goals' column

# Create total_goals if score columns exist
# df_clean['total_goals'] = df_clean['home_score'] + df_clean['away_score']

# Binning into categories
def categorize_goals(total):
    if pd.isna(total):
        return 'Unknown'
    elif total <= 1:
        return 'Low Scoring'
    elif total <= 3:
        return 'Normal'
    elif total <= 5:
        return 'High Scoring'
    else:
        return 'Thriller'

# df_clean['goal_category'] = df_clean['total_goals'].apply(categorize_goals)

print("Goal categorization function defined")

### 3.2 Encoding Categorical Variables

In [ ]:
# Label Encoding
le = LabelEncoder()

# Example: Encode competition type
# df_clean['competition_encoded'] = le.fit_transform(df_clean['competition'])

# One-Hot Encoding
# df_encoded = pd.get_dummies(df_clean, columns=['competition', 'venue'])

print("Encoding methods ready")

### 3.3 Understanding Data Characteristics

In [ ]:
# Statistical characteristics for numeric columns
def analyze_distribution(series):
    """Analyze distribution of a numeric series."""
    return {
        'Mean': series.mean(),
        'Median': series.median(),
        'Std Dev': series.std(),
        'Variance': series.var(),
        'Skewness': series.skew(),
        'Kurtosis': series.kurtosis(),
        'Min': series.min(),
        'Max': series.max()
    }

# Apply to numeric columns
for col in numeric_cols[:3]:  # First 3 numeric columns
    if col in df_clean.columns:
        print(f"\n{col}:")
        stats_dict = analyze_distribution(df_clean[col].dropna())
        for stat, value in stats_dict.items():
            print(f"  {stat}: {value:.2f}")

---

## 4. Normalization & Standardization

In [ ]:
# Min-Max Normalization (scales to 0-1)
min_max_scaler = MinMaxScaler()

# Z-Score Standardization (mean=0, std=1)
standard_scaler = StandardScaler()

# Example usage:
# df_normalized = df_clean.copy()
# df_normalized[numeric_cols] = min_max_scaler.fit_transform(df_clean[numeric_cols])

# df_standardized = df_clean.copy()
# df_standardized[numeric_cols] = standard_scaler.fit_transform(df_clean[numeric_cols])

print("Scalers configured")

---

## 5. Outlier Detection

### 5.1 Z-Score Method

In [ ]:
def detect_outliers_zscore(data, column, threshold=3):
    """Detect outliers using Z-Score method."""
    z_scores = np.abs(stats.zscore(data[column].dropna()))
    outlier_mask = z_scores > threshold
    
    print(f"Z-Score Outlier Detection for '{column}':")
    print(f"  Threshold: {threshold}")
    print(f"  Outliers found: {outlier_mask.sum()}")
    
    return outlier_mask

# Example:
# outliers = detect_outliers_zscore(df_clean, 'total_goals')

### 5.2 IQR Method

In [ ]:
def detect_outliers_iqr(data, column):
    """Detect outliers using IQR method."""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    
    print(f"IQR Outlier Detection for '{column}':")
    print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Outliers found: {len(outliers)}")
    
    return outliers

# Example:
# outliers = detect_outliers_iqr(df_clean, 'attendance')

### 5.3 K-Means Clustering for Outliers

In [ ]:
def detect_outliers_kmeans(data, columns, n_clusters=3):
    """Detect outliers using K-Means clustering."""
    # Prepare data
    X = data[columns].dropna()
    
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Fit K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    
    # Calculate distance to cluster centers
    distances = kmeans.transform(X_scaled).min(axis=1)
    
    # Outliers are points far from all centers
    threshold = np.percentile(distances, 95)
    outlier_mask = distances > threshold
    
    print(f"K-Means Outlier Detection:")
    print(f"  Clusters: {n_clusters}")
    print(f"  Distance threshold (95th percentile): {threshold:.2f}")
    print(f"  Outliers found: {outlier_mask.sum()}")
    
    return outlier_mask, clusters

# Example:
# outliers, clusters = detect_outliers_kmeans(df_clean, ['home_score', 'away_score'])

---

## 6. Feature Engineering

Create new meaningful features.

In [ ]:
def engineer_features(df):
    """Create new features from existing data."""
    df_new = df.copy()
    
    # Example features (adjust column names as needed)
    # Total goals
    # df_new['total_goals'] = df_new['home_score'] + df_new['away_score']
    
    # Goal difference
    # df_new['goal_diff'] = df_new['home_score'] - df_new['away_score']
    
    # Result category
    # df_new['result'] = df_new['goal_diff'].apply(
    #     lambda x: 'Home Win' if x > 0 else ('Away Win' if x < 0 else 'Draw')
    # )
    
    # Decade
    # df_new['decade'] = (df_new['year'] // 10) * 10
    
    # Is high scoring?
    # df_new['high_scoring'] = df_new['total_goals'] > 4
    
    return df_new

print("Feature engineering function defined")

---

## 7. Save Cleaned Data

In [ ]:
# Save the cleaned dataset
output_path = '../data/processed/nb02_cleaned.csv'
df_clean.to_csv(output_path, index=False)
print(f"Saved cleaned data to: {output_path}")
print(f"Final shape: {df_clean.shape}")

---

## Next Steps

Continue to:
- `03_eda.ipynb` - Exploratory Data Analysis
- `04_ml_models.ipynb` - Machine Learning Models